# Generate Supplementary Tables (S3-S6) and Data-Specific Reports

This notebook cell:
- reads each dataset from `raw_files/`
- writes output TSV files into `files/` as `Supplementary_Table3.tsv`, `Supplementary_Table4.tsv`, ...
- creates a data-spec report in Markdown with variable and missing-data details for each generated table.

In [2]:
from pathlib import Path
import pandas as pd
import yaml


def find_base_dir() -> Path:
    # Try current dir and parents first.
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / "raw_files").exists():
            return candidate

    # Fallback to this repository's expected location.
    fallback = cwd / "analysis" / "supplementary_data"
    if (fallback / "raw_files").exists():
        return fallback

    raise FileNotFoundError(
        "Could not locate 'raw_files' directory. Run this notebook from the project root or analysis/supplementary_data."
    )


base_dir = find_base_dir()
raw_dir = base_dir / "raw_files"
out_dir = base_dir / "files"
out_dir.mkdir(parents=True, exist_ok=True)


def read_dataset(path: Path) -> pd.DataFrame:
    suffix = path.suffix.lower()
    if suffix == ".tsv":
        return pd.read_csv(path, sep="\t")
    if suffix == ".csv":
        return pd.read_csv(path)
    if suffix in {".xlsx", ".xls"}:
        return pd.read_excel(path)
    if suffix == ".parquet":
        return pd.read_parquet(path)
    raise ValueError(f"Unsupported file format: {path.name}")


def load_variable_dictionary(dict_path: Path) -> dict:
    if not dict_path or not dict_path.exists():
        return {}
    with dict_path.open("r", encoding="utf-8") as f:
        return yaml.safe_load(f) or {}


def resolve_variable_dictionary(table_stem: str) -> tuple[Path | None, str]:
    final_path = out_dir / f"{table_stem}_variable_dictionary.yaml"
    review_path = out_dir / f"{table_stem}_variable_dictionary.reviewed.yaml"
    draft_path = out_dir / f"{table_stem}_variable_dictionary.draft.yaml"

    # During drafting, prefer the editable review copy over older final files.
    if review_path.exists():
        return review_path, "review_copy"
    if draft_path.exists():
        return draft_path, "draft"
    if final_path.exists():
        return final_path, "final"
    return None, "missing"


def _clean_meta_value(value: object) -> str:
    if value is None:
        return "n/a"
    text = str(value).strip()
    return text if text else "n/a"


def _truncate(text: str, max_len: int) -> str:
    if len(text) <= max_len:
        return text
    return text[: max_len - 3].rstrip() + "..."


def _compact_levels(text: str, max_items: int = 5) -> str:
    # Compact long categorical level sets such as "{'a', 'b', ...}".
    stripped = text.strip()
    if not (stripped.startswith("{") and stripped.endswith("}")):
        return text

    inner = stripped[1:-1].strip()
    if not inner:
        return "{}"

    items = [part.strip() for part in inner.split(",") if part.strip()]
    if len(items) <= max_items:
        return text

    preview = ", ".join(items[:max_items])
    return "{" + preview + f", ...}} ({len(items)} levels)"


def format_variable_table(df: pd.DataFrame, metadata: dict) -> list[str]:
    lines = [
        "| Variable | Description | Unit | Value labels / ranges |",
        "| --- | --- | --- | --- |",
    ]
    for col in df.columns:
        meta = metadata.get(col, {})
        description = _truncate(_clean_meta_value(meta.get("description")), 120)
        unit = _truncate(_clean_meta_value(meta.get("unit")), 40)
        value_labels = _clean_meta_value(
            meta.get("value_labels", meta.get("allowed_range"))
        )
        value_labels = _compact_levels(value_labels)
        value_labels = _truncate(value_labels, 120)
        lines.append(f"| {col} | {description} | {unit} | {value_labels} |")
    return lines


def summarize_missing_data_codes(df: pd.DataFrame) -> list[str]:
    code_counts: dict[str, int] = {}

    # Common explicit missing-value tokens in string/object columns.
    common_codes = {
        "none",
        "unknown",
        "not known",
        "not provided",
        "not collected",
        "not applicable",
        "missing",
        "n/a",
        "na",
        "null",
        "nan",
    }

    for col in df.columns:
        series = df[col]
        if not (
            pd.api.types.is_object_dtype(series)
            or pd.api.types.is_string_dtype(series)
            or isinstance(series.dtype, pd.CategoricalDtype)
        ):
            continue

        text_values = series.dropna().astype(str).str.strip()
        if text_values.empty:
            continue

        lowered = text_values.str.lower()
        for code in common_codes:
            hits = int((lowered == code).sum())
            if hits:
                code_counts[code] = code_counts.get(code, 0) + hits

    lines = []
    for code in sorted(code_counts):
        lines.append(f"  - '{code}': found {code_counts[code]} value(s)")

    nan_count = int(df.isna().sum().sum())
    lines.append(f"  - NaN/blank parsed by pandas: found {nan_count} value(s)")
    return lines


def summarize_specialized_formats(df: pd.DataFrame) -> str:
    datetime_cols = [
        col
        for col in df.columns
        if pd.api.types.is_datetime64_any_dtype(df[col])
    ]

    if datetime_cols:
        return "datetime columns: " + ", ".join(datetime_cols)
    return "none specified"


raw_files = sorted([p for p in raw_dir.iterdir() if p.is_file()])
if not raw_files:
    raise FileNotFoundError(f"No input files found in: {raw_dir}")

if len(raw_files) != 4:
    print(
        f"Warning: expected 4 files for S3-S6, but found {len(raw_files)}. Numbering will continue from S3."
    )

report_lines = []
used_draft_sources = False

for idx, input_path in enumerate(raw_files, start=3):
    df = read_dataset(input_path)

    out_name = f"Supplementary_Table{idx}.tsv"
    out_path = out_dir / out_name
    df.to_csv(out_path, sep="\t", index=False)

    metadata_path, metadata_kind = resolve_variable_dictionary(input_path.stem)
    metadata = load_variable_dictionary(metadata_path)
    if metadata_kind != "final":
        used_draft_sources = True

    report_lines.append(f"# DATA-SPECIFIC INFORMATION FOR: [{out_name}]")
    report_lines.append("")
    report_lines.append(f"* Number of variables: {len(df.columns)}")
    report_lines.append(f"* Number of cases/rows: {len(df)}")
    report_lines.append(
        "* Variable List: variable name, description, unit, and value labels/ranges"
    )
    report_lines.extend(format_variable_table(df, metadata))
    report_lines.append("* Missing data codes: *list code/symbol and definition*")
    report_lines.extend(summarize_missing_data_codes(df))
    report_lines.append(
        f"* Specialized formats or other abbreviations used: {summarize_specialized_formats(df)}"
    )
    report_lines.append("")

report_name = (
    "Supplementary_Data_Spec_Report.draft.md"
    if used_draft_sources
    else "Supplementary_Data_Spec_Report.md"
)
report_path = out_dir / report_name
report_path.write_text("\n".join(report_lines), encoding="utf-8")

print(f"Generated {len(raw_files)} supplementary table(s) in: {out_dir}")
print(f"Report written to: {report_path}")

Generated 4 supplementary table(s) in: /home/tobamo/analize/project-tobamo/analysis/supplementary_data/files
Report written to: /home/tobamo/analize/project-tobamo/analysis/supplementary_data/files/Supplementary_Data_Spec_Report.draft.md
